# 신재용_머신러닝프로젝트.ipynb

필요 라이브러리 설치

pip install pandas numpy matplotlib seaborn scikit-learn openpyxl streamlit joblib notebook setuptools ydata-profiling


실행 순서

1. `python preprocess.py`
2. 이 노트북에서 **Run All**
3. `streamlit run 신재용_머신러닝프로젝트.py`

이 노트북은 모델 학습, 평가, 결과 저장, `best_model.joblib` 및 `model_metadata.pkl` 저장을 담당합니다.


In [ ]:
from __future__ import annotations

import math
import pickle
from pathlib import Path
from typing import Dict, List

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from preprocess import (
    TARGET_COL,
    MODEL_OUTPUT_DIR,
    REFORMED_DIR,
    REPORT_DIR,
    ensure_preprocessed_files,
    load_preprocessed_split_data,
    log,
)

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
pd.options.display.float_format = "{:.4f}".format

MODEL_JOBLIB_PATH = MODEL_OUTPUT_DIR / "best_model.joblib"
MODEL_METADATA_PKL_PATH = MODEL_OUTPUT_DIR / "model_metadata.pkl"


In [ ]:
def get_candidate_specs() -> List[Dict]:
    return [
        {
            "name": "linear_regression_log",
            "factory": lambda: LinearRegression(),
            "use_log_target": True,
        },
        {
            "name": "ridge_alpha_1_log",
            "factory": lambda: Pipeline([
                ("scaler", StandardScaler()),
                ("model", Ridge(alpha=1.0)),
            ]),
            "use_log_target": True,
        },
        {
            "name": "ridge_alpha_10_log",
            "factory": lambda: Pipeline([
                ("scaler", StandardScaler()),
                ("model", Ridge(alpha=10.0)),
            ]),
            "use_log_target": True,
        },
        {
            "name": "random_forest_log",
            "factory": lambda: RandomForestRegressor(
                n_estimators=500,
                max_depth=12,
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1,
            ),
            "use_log_target": True,
        },
        {
            "name": "gradient_boosting_log",
            "factory": lambda: GradientBoostingRegressor(
                n_estimators=300,
                learning_rate=0.05,
                max_depth=3,
                subsample=0.9,
                random_state=42,
            ),
            "use_log_target": True,
        },
    ]


def fit_candidate_model(spec: Dict, X: pd.DataFrame, y: pd.Series):
    model = spec["factory"]()
    if spec["use_log_target"]:
        model.fit(X, np.log1p(y))
    else:
        model.fit(X, y)
    return model


def predict_candidate_model(spec: Dict, model, X: pd.DataFrame) -> np.ndarray:
    pred = model.predict(X)
    if spec["use_log_target"]:
        pred = np.expm1(pred)
    pred = np.clip(np.asarray(pred, dtype=float), 0, None)
    return pred


def calc_regression_metrics(y_true, y_pred) -> Dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    safe_true = np.where(y_true == 0, np.nan, y_true)
    mape = np.nanmean(np.abs((y_true - y_pred) / safe_true)) * 100
    return {
        "r2": float(r2_score(y_true, y_pred)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mape": float(mape) if not math.isnan(mape) else np.nan,
        "bias": float(np.mean(y_pred - y_true)),
    }


def extract_feature_importance(model, feature_names: List[str]) -> pd.DataFrame:
    inner = model.named_steps["model"] if isinstance(model, Pipeline) else model

    if hasattr(inner, "coef_"):
        importance = np.abs(np.ravel(inner.coef_))
        df = pd.DataFrame({
            "feature": feature_names,
            "importance": importance,
            "signed_value": np.ravel(inner.coef_),
            "importance_type": "abs_coefficient",
        })
        return df.sort_values("importance", ascending=False).reset_index(drop=True)

    if hasattr(inner, "feature_importances_"):
        importance = np.asarray(inner.feature_importances_, dtype=float)
        df = pd.DataFrame({
            "feature": feature_names,
            "importance": importance,
            "signed_value": importance,
            "importance_type": "feature_importance",
        })
        return df.sort_values("importance", ascending=False).reset_index(drop=True)

    return pd.DataFrame({
        "feature": feature_names,
        "importance": np.nan,
        "signed_value": np.nan,
        "importance_type": "not_supported",
    })


def evaluate_model_candidates():
    data = load_preprocessed_split_data()
    X_train = data["X_train"]
    X_val = data["X_val"]
    X_test = data["X_test"]
    y_train = data["y_train"]
    y_val = data["y_val"]
    y_test = data["y_test"]

    rows = []
    fitted_models = {}

    for spec in get_candidate_specs():
        model = fit_candidate_model(spec, X_train, y_train)
        fitted_models[spec["name"]] = (spec, model)

        train_pred = predict_candidate_model(spec, model, X_train)
        val_pred = predict_candidate_model(spec, model, X_val)
        test_pred = predict_candidate_model(spec, model, X_test)

        train_metrics = calc_regression_metrics(y_train, train_pred)
        val_metrics = calc_regression_metrics(y_val, val_pred)
        test_metrics = calc_regression_metrics(y_test, test_pred)

        rows.append({
            "model_name": spec["name"],
            "train_r2": train_metrics["r2"],
            "val_r2": val_metrics["r2"],
            "test_r2": test_metrics["r2"],
            "train_mae": train_metrics["mae"],
            "val_mae": val_metrics["mae"],
            "test_mae": test_metrics["mae"],
            "train_rmse": train_metrics["rmse"],
            "val_rmse": val_metrics["rmse"],
            "test_rmse": test_metrics["rmse"],
            "val_mape": val_metrics["mape"],
            "test_mape": test_metrics["mape"],
            "val_bias": val_metrics["bias"],
            "test_bias": test_metrics["bias"],
        })

    comparison_df = pd.DataFrame(rows)
    comparison_df = comparison_df.sort_values(by=["val_r2", "val_mae"], ascending=[False, True]).reset_index(drop=True)
    comparison_df.to_csv(MODEL_OUTPUT_DIR / "model_comparison.csv", index=False, encoding="utf-8-sig")

    best_name = comparison_df.iloc[0]["model_name"]
    best_spec, best_eval_model = fitted_models[best_name]
    feature_importance_df = extract_feature_importance(best_eval_model, X_train.columns.tolist())
    feature_importance_df.to_csv(MODEL_OUTPUT_DIR / "best_model_feature_importance.csv", index=False, encoding="utf-8-sig")

    X_train_val = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
    y_train_val = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)
    best_inference_model = fit_candidate_model(best_spec, X_train_val, y_train_val)

    return {
        "comparison_df": comparison_df,
        "best_model_name": best_name,
        "best_spec": best_spec,
        "best_eval_model": best_eval_model,
        "best_inference_model": best_inference_model,
        "feature_importance_df": feature_importance_df,
        **data,
    }


def save_prediction_results(prefix_df: pd.DataFrame, y_true, y_pred, filename: str):
    out = pd.DataFrame({
        "date": prefix_df["date"],
        "actual": y_true,
        "predicted": y_pred,
    })
    out.to_csv(MODEL_OUTPUT_DIR / filename, index=False, encoding="utf-8-sig")
    return out


def save_scatter_plot(y_true, y_pred, title: str, filename: str, show_plot: bool):
    plt.figure(figsize=(10, 6))
    plt.scatter(y_true, y_pred, alpha=0.55)
    min_val = min(np.min(y_true), np.min(y_pred))
    max_val = max(np.max(y_true), np.max(y_pred))
    plt.plot([min_val, max_val], [min_val, max_val], "r--", linewidth=2)
    plt.xlabel("실제 대여건수")
    plt.ylabel("예측 대여건수")
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(MODEL_OUTPUT_DIR / filename, dpi=150)
    if show_plot:
        plt.show()
    else:
        plt.close()


def save_timeseries_plot(result_df: pd.DataFrame, title: str, filename: str, show_plot: bool):
    plt.figure(figsize=(12, 6))
    plt.plot(result_df["date"], result_df["actual"], label="실제 대여건수")
    plt.plot(result_df["date"], result_df["predicted"], label="예측 대여건수")
    plt.xlabel("날짜")
    plt.ylabel("대여건수")
    plt.title(title)
    plt.xticks(rotation=45)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(MODEL_OUTPUT_DIR / filename, dpi=150)
    if show_plot:
        plt.show()
    else:
        plt.close()


def save_model_artifacts(model, metadata: Dict):
    joblib.dump(model, MODEL_JOBLIB_PATH)
    with open(MODEL_METADATA_PKL_PATH, "wb") as f:
        pickle.dump(metadata, f)


In [ ]:
def run_training_pipeline(show_plots: bool = False) -> Dict:
    ensure_preprocessed_files()
    bundle = evaluate_model_candidates()
    best_spec = bundle["best_spec"]
    best_eval_model = bundle["best_eval_model"]
    best_model_name = bundle["best_model_name"]

    X_train = bundle["X_train"]
    X_val = bundle["X_val"]
    X_test = bundle["X_test"]
    y_train = bundle["y_train"]
    y_val = bundle["y_val"]
    y_test = bundle["y_test"]

    train_pred = predict_candidate_model(best_spec, best_eval_model, X_train)
    val_pred = predict_candidate_model(best_spec, best_eval_model, X_val)
    test_pred = predict_candidate_model(best_spec, best_eval_model, X_test)

    train_metrics = calc_regression_metrics(y_train, train_pred)
    val_metrics = calc_regression_metrics(y_val, val_pred)
    test_metrics = calc_regression_metrics(y_test, test_pred)

    train_result = save_prediction_results(bundle["train_df"], y_train, train_pred, "train_prediction_result.csv")
    val_result = save_prediction_results(bundle["val_df"], y_val, val_pred, "val_prediction_result.csv")
    test_result = save_prediction_results(bundle["test_df"], y_test, test_pred, "test_prediction_result.csv")

    save_scatter_plot(y_val, val_pred, f"Validation 실제값 vs 예측값 산점도 ({best_model_name})", "val_scatter.png", show_plots)
    save_scatter_plot(y_test, test_pred, f"Test 실제값 vs 예측값 산점도 ({best_model_name})", "test_scatter.png", show_plots)
    save_timeseries_plot(val_result, f"Validation 날짜별 실제값 vs 예측값 ({best_model_name})", "val_timeseries.png", show_plots)
    save_timeseries_plot(test_result, f"Test 날짜별 실제값 vs 예측값 ({best_model_name})", "test_timeseries.png", show_plots)

    summary_df = pd.DataFrame([
        {"split": "train", **train_metrics},
        {"split": "validation", **val_metrics},
        {"split": "test", **test_metrics},
    ])
    summary_df.insert(0, "model_name", best_model_name)
    summary_df.to_csv(MODEL_OUTPUT_DIR / "best_model_metrics.csv", index=False, encoding="utf-8-sig")

    metadata = {
        "model_name": best_model_name,
        "feature_columns": bundle["X_train"].columns.tolist(),
        "date_origin": bundle["date_origin"],
        "metrics": {
            "train": train_metrics,
            "validation": val_metrics,
            "test": test_metrics,
        },
        "comparison_df_path": str(MODEL_OUTPUT_DIR / "model_comparison.csv"),
        "feature_importance_path": str(MODEL_OUTPUT_DIR / "best_model_feature_importance.csv"),
        "metrics_csv_path": str(MODEL_OUTPUT_DIR / "best_model_metrics.csv"),
    }
    save_model_artifacts(bundle["best_inference_model"], metadata)

    return {
        "model_name": best_model_name,
        "train_metrics": train_metrics,
        "validation_metrics": val_metrics,
        "test_metrics": test_metrics,
        "metadata_path": MODEL_METADATA_PKL_PATH,
        "model_path": MODEL_JOBLIB_PATH,
        "comparison_df": bundle["comparison_df"],
        "feature_importance_df": bundle["feature_importance_df"],
    }


In [ ]:
ensure_preprocessed_files()
result = run_training_pipeline(show_plots=False)

print("\n[학습 완료]")
print(f"- 선택 모델: {result['model_name']}")
print(f"- Validation R²: {result['validation_metrics']['r2']:.4f}")
print(f"- Test R²: {result['test_metrics']['r2']:.4f}")
print(f"- 모델 저장(joblib): {result['model_path']}")
print(f"- 메타데이터 저장(pkl): {result['metadata_path']}")

result["comparison_df"]


In [ ]:
result["feature_importance_df"].head(20)
